# Network Layer: IPv4 and IPv6 Packet Structures
This notebook contains the practical Linux CLI exercises for analyzing IP packet structures.
Execute the cells sequentially to install the required tools, generate network traffic, and parse the resulting packets.


In [ ]:
%%bash
# Install the required packet analysis tools, including the missing ping utility
apt-get update > /dev/null 2>&1
apt-get install -y tcpdump tshark traceroute curl iputils-ping > /dev/null 2>&1
echo "Environment setup complete. Tools installed."

Environment setup complete. Tools installed.


## Part 1: IPv4 Analysis
We will generate standard IPv4 ICMP (ping) and TCP (web) traffic to observe the header fields, including Version, Header Length, ToS, Datagram Length, and Fragmentation.

In [ ]:
%%bash
echo "Capturing 1 standard IPv4 ping packet..."

# 1. Start tcpdump in the background
# We capture exactly 1 packet (-c 1), output in verbose (-v) and hex (-xx)
tcpdump -i any -c 1 -v -xx icmp > ipv4_basic.txt 2>&1 &
sleep 1 # Give tcpdump a second to start listening

# 2. Generate the traffic: Ping Google DNS with a 100-byte payload
ping -c 1 -s 100 8.8.8.8 > /dev/null
sleep 1

# 3. Read the capture file
cat ipv4_basic.txt

Capturing 1 standard IPv4 ping packet...
tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
07:11:00.088155 eth0  Out IP (tos 0x0, ttl 64, id 25347, offset 0, flags [DF], proto ICMP (1), length 128)
    3375ec2e118d > dns.google: ICMP echo request, id 7, seq 1, length 108
	0x0000:  0800 0000 0000 0007 0001 0406 0242 ac1c
	0x0010:  000c 0000 4500 0080 6303 4000 4001 1b42
	0x0020:  ac1c 000c 0808 0808 0800 e9bf 0007 0001
	0x0030:  84e8 dd69 0000 0000 4758 0100 0000 0000
	0x0040:  1011 1213 1415 1617 1819 1a1b 1c1d 1e1f
	0x0050:  2021 2223 2425 2627 2829 2a2b 2c2d 2e2f
	0x0060:  3031 3233 3435 3637 3839 3a3b 3c3d 3e3f
	0x0070:  4041 4243 4445 4647 4849 4a4b 4c4d 4e4f
	0x0080:  5051 5253 5455 5657 5859 5a5b 5c5d 5e5f
	0x0090:  6061 6263
1 packet captured
2 packets received by filter
0 packets dropped by kernel


In [ ]:
%%bash
echo "Capturing fragmented IPv4 traffic..."

# 1. Start tcpdump to capture 2 ICMP packets
tcpdump -i any -c 2 -v -n icmp > ipv4_frag.txt 2>&1 &
sleep 1

# 2. Generate a ping with a 3000-byte payload.
# This vastly exceeds the 1500-byte Ethernet MTU, forcing fragmentation.
ping -c 1 -s 3000 8.8.8.8 > /dev/null
sleep 1

# 3. Read the capture file. Notice the 'id' matches, but the 'offset' and 'flags' change.
cat ipv4_frag.txt

Capturing fragmented IPv4 traffic...
tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
07:11:02.109192 eth0  Out IP (tos 0x0, ttl 64, id 25808, offset 0, flags [+], proto ICMP (1), length 1500)
    172.28.0.12 > 8.8.8.8: ICMP echo request, id 8, seq 1, length 1480
07:11:02.109208 eth0  Out IP (tos 0x0, ttl 64, id 25808, offset 1480, flags [+], proto ICMP (1), length 1500)
    172.28.0.12 > 8.8.8.8: ip-proto-1
2 packets captured
3 packets received by filter
0 packets dropped by kernel


In [ ]:
%%bash
echo "Capturing an expired TTL packet..."

# 1. Start tcpdump listening for ICMP Time Exceeded errors
tcpdump -i any -c 1 -v -n "icmp[icmptype] == icmp-timxceed" > ipv4_ttl.txt 2>&1 &
sleep 1

# 2. Generate a ping to a remote server but artificially set the TTL to 1 (-t 1).
# The very first router will drop it and send back the error.
ping -c 1 -t 1 8.8.8.8 > /dev/null 2>&1
sleep 1

# 3. Read the capture file. Look for the "Time to live exceeded" message.
cat ipv4_ttl.txt

Capturing an expired TTL packet...
tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
07:11:14.131123 eth0  In  IP (tos 0xc0, ttl 64, id 37068, offset 0, flags [none], proto ICMP (1), length 112)
    172.28.0.1 > 172.28.0.12: ICMP time exceeded in-transit, length 92
	IP (tos 0x0, ttl 1, id 36346, offset 0, flags [DF], proto ICMP (1), length 84)
    172.28.0.12 > 8.8.8.8: ICMP echo request, id 9, seq 1, length 64
1 packet captured
1 packet received by filter
0 packets dropped by kernel


In [ ]:
%%bash
echo "Capturing HTTP traffic to observe Protocol 6 (TCP)..."

# 1. Start tcpdump filtering strictly for port 80
tcpdump -i any -c 1 -v -n -nn tcp port 80 > ipv4_proto.txt 2>&1 &
sleep 1

# 2. Generate a web request
curl -s http://neverssl.com > /dev/null
sleep 1

# 3. Read the capture file. Look for "proto TCP (6)" and the Source > Dest IPs.
cat ipv4_proto.txt

Capturing HTTP traffic to observe Protocol 6 (TCP)...
tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
07:11:16.179327 eth0  Out IP (tos 0x0, ttl 64, id 52836, offset 0, flags [DF], proto TCP (6), length 60)
    172.28.0.12.40088 > 34.223.124.45.80: Flags [S], cksum 0x4b63 (incorrect -> 0x65a3), seq 1293707922, win 64240, options [mss 1460,sackOK,TS val 1067976696 ecr 0,nop,wscale 7], length 0
1 packet captured
2 packets received by filter
0 packets dropped by kernel


## Part 2: IPv6 Analysis
Because Colab environments often lack external IPv6 routing, we will use the IPv6 loopback interface (`lo`) and address (`::1`) to reliably generate and capture IPv6 datagrams.

In [ ]:
%%bash
echo "Capturing 1 standard IPv6 ping packet..."

# 1. Start tcpdump listening on the loopback interface for IPv6 traffic
tcpdump -i lo -c 1 -v -xx ip6 > ipv6_basic.txt 2>&1 &
sleep 1

# 2. Generate traffic using ping6 with a 50-byte payload
ping6 -c 1 -s 50 ::1 > /dev/null
sleep 1

# 3. Read the capture file. Look for "plen 58" and "hlim 64".
cat ipv6_basic.txt

Capturing 1 standard IPv6 ping packet...
tcpdump: listening on lo, link-type EN10MB (Ethernet), snapshot length 262144 bytes
07:11:23.706798 IP6 (flowlabel 0xfc105, hlim 64, next-header ICMPv6 (58) payload length: 58) localhost > localhost: [icmp6 sum ok] ICMP6, echo request, id 10, seq 1
	0x0000:  0000 0000 0000 0000 0000 0000 86dd 600f
	0x0010:  c105 003a 3a40 0000 0000 0000 0000 0000
	0x0020:  0000 0000 0001 0000 0000 0000 0000 0000
	0x0030:  0000 0000 0001 8000 0030 000a 0001 9be8
	0x0040:  dd69 0000 0000 d9c8 0a00 0000 0000 1011
	0x0050:  1213 1415 1617 1819 1a1b 1c1d 1e1f 2021
	0x0060:  2223 2425 2627 2829 2a2b 2c2d 2e2f 3031
1 packet captured
4 packets received by filter
0 packets dropped by kernel


In [ ]:
%%bash
echo "Using Tshark for deep IPv6 protocol tree expansion..."

# 1. Start tshark in the background.
# We route stderr (2) to /dev/null to hide harmless Docker capability warnings.
tshark -i lo -O ipv6 -c 1 -Y "icmpv6" -a duration:5 > ipv6_tshark.txt 2>/dev/null &
sleep 2

# 2. Generate IPv6 traffic (now that ping6 is installed!)
ping6 -c 1 ::1 > /dev/null
sleep 2

# 3. Read the capture file. Look for the "Traffic Class" and "Next Header" fields.
cat ipv6_tshark.txt

Using Tshark for deep IPv6 protocol tree expansion...


In [ ]:
%%bash
echo "Capturing highly verbose IPv6 traffic to view Flow Labels..."

# 1. Start tcpdump with -vv (very verbose) to ensure the flow label is printed
tcpdump -i lo -c 1 -vv -n ip6 > ipv6_flow.txt 2>&1 &
sleep 1

# 2. Generate IPv6 traffic
ping6 -c 1 ::1 > /dev/null
sleep 1

# 3. Read the capture file. Look for "flowlabel" and the 128-bit ::1 addresses.
cat ipv6_flow.txt

Capturing highly verbose IPv6 traffic to view Flow Labels...
tcpdump: listening on lo, link-type EN10MB (Ethernet), snapshot length 262144 bytes
07:11:29.748821 IP6 (flowlabel 0xfc105, hlim 64, next-header ICMPv6 (58) payload length: 64) ::1 > ::1: [icmp6 sum ok] ICMP6, echo request, id 12, seq 1
1 packet captured
4 packets received by filter
0 packets dropped by kernel
